# Chapter 15: Perfect Hash Filters (BPHF)

Companion notebook for **Chapter 15** of *Bernoulli Types: Two-Sided Error in Approximate Membership, Retrieval, and Computation*. Source files: `book/chapters/ch15/`.

Chapter 14 gave two constructions on the FNR-zero arm of the perfect-filter line. The **Bernoulli hash function** (`def:bhf`) stores one salt found by search; it meets the entropy floor to leading order but builds in time exponential in $n$. The **perfect hash filter** (`def:phf`) addresses a table of $r$-bit fingerprints through a minimal perfect hash $\phi : A \to [n]$, with $r = \lceil -\log_2 \fprate \rceil$; it has $\fnrate = 0$ and $\fprate \le 2^{-r}$, costs $r$ bits per element for the table plus at least $\log_2 e$ for the perfect hash (`prop:phf-space`), and is the standard static construction described by Dietzfelbinger and Pagh (2008, Sec. 1). That second construction is the skeleton of this chapter. The **Bernoulli Perfect Hash Filter (BPHF)** (`def:bphf`) keeps $\phi$ and stores an $r$-bit *rank* per slot instead of a fingerprint, with $r = \lceil \log_2 n \rceil$.

The chapter's honest accounting is the point of this notebook, so every claim is checked with an `assert` (a clean run means all checks passed):

1. **General query model.** A non-member presenting a recomputable hash-based rank matches the stored rank with probability about $2^{-r} \approx 1/n$. The bare construction *is* the perfect hash filter at width $r = \lceil \log_2 n \rceil$: $\fnrate = 0$, $\fprate \approx 2^{-r}$. The rank relabel alone does not flip the error axis.
2. **Witness model.** When a non-member carries no valid rank to present, the false-positive rate is exactly 0. The guarantee belongs to the query model, not the table (`rem:witness-is-query-model`): a constant or low-entropy public rank is forgeable.
3. **Exact corner.** With a true perfect hash every member owns its slot, so $\fnrate = 0$; rank collisions among members cause no misses.
4. **$k$-perfect dial.** A $k$-perfect hash into $m < n$ slots, $o$ of them occupied, displaces exactly $n - o$ members: $\fnrate = 1 - o/n \ge 1 - m/n$, while $\fprate$ stays 0 in the witness model. Three cautions: the misses are dependent within slots, occupancy formulas must use the construction's conditioning, and in the fingerprint reading a displaced member is rescued with probability $2^{-r}$.
5. **Ranked search.** Top-$k$ retrieval is the witness model: exact at full resolution, about $k(1 - m/n)$ expected misses under the $k$-perfect dial (`prop:bphf-ranked-search`), never an off-corpus result.
6. **Space.** Both structures pay the same addressing cost $c \ge \log_2 e$; the slot widths $\lceil \log_2 n \rceil$ (BPHF) and $\lceil -\log_2 \fprate \rceil$ (PHF) cross at $\fprate = 1/n$.

In [1]:
import hashlib
import math
from fractions import Fraction
from itertools import product

import numpy as np

from bernoulli.maps import PerfectFilter

rng = np.random.default_rng(15)
print('NumPy', np.__version__)

def _h(salt, key, mod):
    """Salted SHA-256 of key, reduced into [0, mod). Distinct salts give independent functions."""
    d = hashlib.sha256((salt + '|' + str(key)).encode()).digest()
    return int.from_bytes(d[:8], 'big') % mod

def rho_hash(key, n):
    """Recomputable rank in [0, n): the witness a non-member could present in the general model.
    Independent of the slot hash phi (different salt)."""
    return _h('rho', key, n)

n = 1000
r = int(math.ceil(math.log2(n)))
print(f'n = {n}, r = ceil(log2 n) = {r} bits')
assert r == 10

NumPy 2.5.4
n = 1000, r = ceil(log2 n) = 10 bits


## The construction

The perfect hash `phi` is a bijection of the members onto `[n]` (a minimal perfect hash; here an index map). Its extension to non-members hashes into `[n]` with a salt independent of the rank, so a non-member lands in *some* member's slot. The rank table `R` stores `rho(a)` at `phi(a)`. The query accepts `x` when `R[phi(x)] == rho(x)`. This is the layout of the perfect hash filter (`fig:phf-layout`) with one $r$-bit rank per slot in place of a fingerprint.

In [2]:
class BPHF:
    """Minimal-perfect-hash + rank table. One r-bit value per slot; the rank is the witness.
    The slot hash (phi) and the rank hash are independent (different salts)."""
    def __init__(self, members, ranks):
        self.phi = {m: i for i, m in enumerate(members)}   # minimal perfect hash: bijection onto [n]
        self.n = len(members)
        self.R = [None] * self.n
        for m in members:
            self.R[self.phi[m]] = ranks[m]

    def _phi_ext(self, x):
        if x in self.phi:
            return self.phi[x]
        return _h('phi', x, self.n)    # non-member: independent slot hash, lands in some occupied slot

    def query(self, x, rank_of):
        return self.R[self._phi_ext(x)] == rank_of(x)

members = [f'doc{i:04d}' for i in range(n)]
ranks = {m: i for i, m in enumerate(members)}   # distinct ranks 0..n-1 (a permutation)
bphf = BPHF(members, ranks)
print('built BPHF over', n, 'members,', r, 'bits per slot')

built BPHF over 1000 members, 10 bits per slot


## Claims 1 and 2: the bare construction is the perfect hash filter; the witness model gives FPR = 0

In the general model a non-member presents `rho_hash(x, n)`, independent of its slot, so the comparison succeeds with probability about `1/n`, the fingerprint-collision probability $2^{-r}$ of `prop:phf-fpr`. For comparison we query the library perfect hash filter (`bernoulli.maps.PerfectFilter`, which the package's docstrings still call the BHF) at the same width $r = 10$: renaming a fingerprint a rank changes nothing the query can observe. In the witness model a non-member has no valid rank to present (modelled by a sentinel that never matches).

In [3]:
Q = 100_000

# General model: a non-member presents a hash-based rank in [0, n), independent of its slot.
fp_general = sum(bphf.query(f'x{j}', lambda z: rho_hash(z, n)) for j in range(Q))
expected = Q / n
print(f'general-model false positives: {fp_general}  (expected ~ Q/n = {expected:.0f}, i.e. FPR ~ 1/n ~ 2^-r)')
assert 0.5 * expected < fp_general < 2.0 * expected, 'general-model FPR should be ~1/n (perfect hash filter behaviour)'

# The perfect hash filter at the same slot width: fingerprints instead of ranks.
members_bytes = [m.encode() for m in members]
pf = PerfectFilter(members_bytes, fingerprint_bits=r)
fp_pf = sum(pf.contains(f'x{j}'.encode()) for j in range(Q))
print(f'perfect hash filter at r = {r}: {fp_pf} false positives  (2^-r Q = {Q * 2.0 ** -r:.0f}); '
      f'misses on members: {sum(not pf.contains(m) for m in members_bytes)}')
assert 0.5 * Q * 2.0 ** -r < fp_pf < 2.0 * Q * 2.0 ** -r
assert all(pf.contains(m) for m in members_bytes)

# Witness model: a non-member has no valid rank (sentinel -1 never equals a stored rank in [0, n)).
fp_witness = sum(bphf.query(f'x{j}', lambda z: -1) for j in range(Q))
print(f'witness-model false positives: {fp_witness}')
assert fp_witness == 0, 'witness-model FPR must be exactly 0'

general-model false positives: 95  (expected ~ Q/n = 100, i.e. FPR ~ 1/n ~ 2^-r)


perfect hash filter at r = 10: 114 false positives  (2^-r Q = 98); misses on members: 0
witness-model false positives: 0


### The guarantee lives in the query model (`rem:witness-is-query-model`)

$\fprate = 0$ holds only when $\rho$ is a witness that members possess and non-members cannot forge. A degenerate assignment, $\rho(a) = 0$ for every member, writes zero into every slot, keeps $\fnrate = 0$, and admits every non-member that presents zero. A low-entropy public $\rho$ that anyone can evaluate, such as a coarse four-way length bucket, is forgeable too: a non-member presenting its own bucket matches the stored one with probability about $1/4$. The table is identical in all three cases; only what non-members can present changes.

In [4]:
zero_ranks = {m: 0 for m in members}
bphf_zero = BPHF(members, zero_ranks)
assert all(bphf_zero.query(m, lambda z: 0) for m in members), 'FNR stays 0'
fp_zero = sum(bphf_zero.query(f'x{j}', lambda z: 0) for j in range(Q))
print(f'constant rank 0: {fp_zero}/{Q} non-members presenting 0 are accepted')
assert fp_zero == Q


def length_bucket(key):
    return _h('len', key, 4)    # public, low-entropy, recomputable by anyone


bphf_bucket = BPHF(members, {m: length_bucket(m) for m in members})
assert all(bphf_bucket.query(m, length_bucket) for m in members)
fp_bucket = sum(bphf_bucket.query(f'x{j}', length_bucket) for j in range(Q)) / Q
print(f'public 4-way bucket rank: non-member acceptance rate {fp_bucket:.4f}  (about 1/4)')
assert abs(fp_bucket - 0.25) < 0.01

constant rank 0: 100000/100000 non-members presenting 0 are accepted
public 4-way bucket rank: non-member acceptance rate 0.2502  (about 1/4)


## Claim 3: a true perfect hash gives FNR = 0, and rank collisions do not cause misses

In [5]:
# Every member owns its slot and reads back its own rank.
misses = sum(not bphf.query(m, lambda z: ranks[z]) for m in members)
print(f'member misses (true perfect hash): {misses}')
assert misses == 0, 'true perfect hash must miss no member'

# Force a rank collision: doc0001 takes doc0000's rank. Both must still be retrieved (distinct slots).
tie = dict(ranks); tie['doc0001'] = tie['doc0000']
bphf_tie = BPHF(members, tie)
both = bphf_tie.query('doc0000', lambda z: tie[z]) and bphf_tie.query('doc0001', lambda z: tie[z])
print(f'both rank-colliding members retrieved: {both}')
assert both, 'a rank collision must NOT cause a false negative under a perfect hash'

member misses (true perfect hash): 0
both rank-colliding members retrieved: True


## Claim 4: the $k$-perfect dial gives $\fnrate = 1 - o/n \ge 1 - m/n$ at $\fprate = 0$

A $k$-perfect hash lets up to $k$ members share a slot (Belazzougui et al. 2009). A shared slot holds only one member's rank; the members that lose the contention read another member's rank, fail the comparison, and are reported as non-members, while a non-member still cannot present a witness. If the construction occupies $o \le m$ slots, exactly $n - o$ members are displaced (`eq:bphf-fnr`):
$$\fnrate = 1 - \frac{o}{n} \;\ge\; 1 - \frac{m}{n},$$
with equality when every slot is occupied. A balanced packing (member `i` to slot `i % m`, first arrival kept) occupies every slot and attains $1 - m/n$; a random placement leaves some slots empty and misses more.

In [6]:
def kperfect_build(n, m, slot_of):
    # Assign members to slots; the first arrival in a slot keeps it. Returns (displaced, occupied).
    owner = {}
    displaced = []
    for i in range(n):
        s = slot_of(i)
        if s in owner:
            displaced.append(i)          # slot already taken: this member is crowded out
        else:
            owner[s] = i
    return displaced, len(owner)


for frac in (1.00, 0.95, 0.90, 0.80):
    m = int(round(frac * n))
    displaced, occupied = kperfect_build(n, m, lambda i: i % m)
    fnr = len(displaced) / n
    print(f'balanced, m = {m:4d} ({frac:.2f} n): occupied {occupied}, FNR = {fnr:.3f}   1 - m/n = {1 - m / n:.3f}   FPR = 0')
    assert occupied == m and abs(fnr - (1 - m / n)) < 1e-12, 'fully occupied: FNR = 1 - m/n'

rng_place = np.random.default_rng(1515)
m = 900
slots = rng_place.integers(0, m, size=n)
displaced, occupied = kperfect_build(n, m, lambda i: int(slots[i]))
fnr = len(displaced) / n
print(f'random,   m = {m:4d} (0.90 n): occupied {occupied}, FNR = {fnr:.3f} = 1 - o/n = {1 - occupied / n:.3f} > 1 - m/n = {1 - m / n:.3f}')
assert abs(fnr - (1 - occupied / n)) < 1e-12 and fnr > 1 - m / n

balanced, m = 1000 (1.00 n): occupied 1000, FNR = 0.000   1 - m/n = 0.000   FPR = 0
balanced, m =  950 (0.95 n): occupied 950, FNR = 0.050   1 - m/n = 0.050   FPR = 0
balanced, m =  900 (0.90 n): occupied 900, FNR = 0.100   1 - m/n = 0.100   FPR = 0
balanced, m =  800 (0.80 n): occupied 800, FNR = 0.200   1 - m/n = 0.200   FPR = 0
random,   m =  900 (0.90 n): occupied 591, FNR = 0.409 = 1 - o/n = 0.409 > 1 - m/n = 0.100


### Three cautions on the dial

1. **The misses are not independent across members.** They are decided slot by slot, exactly one of the members sharing a slot surviving, so the member side does not satisfy element-wise independence (Axiom 1). The fraction $1 - o/n$ is a property of the build, not a per-element coin: with a balanced packing the number of misses is exactly $n - m$ in every build, where independent coins at the same rate would scatter binomially.
2. **Occupancy formulas must use the construction's conditioning.** A build that rejects placements with an overfull slot has, at $m = n = 2$ and $k = 1$, no displaced members at all, whereas the unconditional occupancy law gives an expected displaced fraction of $1/4$.
3. **The fingerprint reading rescues some displaced members.** If the same skeleton stores fingerprints, a displaced member is accepted whenever its fingerprint happens to equal the slot owner's, so its false-negative rate is the displaced fraction times $1 - 2^{-r}$. With one-bit fingerprints and $k = 2$, a displacement probability of $1/4$ gives a membership false-negative rate of $1/8$.

Dropping members outright, as in the lossy dictionaries of Pagh and Rodler (2001), achieves a given $\fnrate$ in fewer bits per retained member; the $k$-perfect variant buys construction simplicity rather than space.

In [7]:
# Caution 1: within a slot exactly one member survives; misses are slot-level events.
members_of = {}
for i in range(n):
    members_of.setdefault(int(slots[i]), []).append(i)
pairs = [ms for ms in members_of.values() if len(ms) == 2]
displaced_set = set(displaced)
assert all(sum(i in displaced_set for i in ms) == 1 for ms in pairs)
assert all(sum(i in displaced_set for i in ms) == len(ms) - 1 for ms in members_of.values())
print(f'{len(pairs)} two-member slots: in every one exactly one member is displaced '
      f'(miss indicators sum to 1, correlation -1)')
p_miss = 1 - 900 / n
print(f'balanced packing at m = 900: misses = {n - 900} in every build (variance 0); '
      f'independent coins at {p_miss:.2f} would have variance {n * p_miss * (1 - p_miss):.0f}')

# Caution 2: m = n = 2, k = 1. Enumerate the four equally likely placements.
placements = list(product(range(2), repeat=2))
frac_displaced = [Fraction(2 - len(set(pl)), 2) for pl in placements]
unconditional = sum(frac_displaced) / len(placements)
accepted = [f for pl, f in zip(placements, frac_displaced) if len(set(pl)) == 2]   # reject overfull slots
conditional = sum(accepted) / len(accepted)
print(f'm = n = 2, k = 1: unconditional expected displaced fraction = {unconditional}; '
      f'conditional on the build rejecting overfull slots = {conditional}')
assert unconditional == Fraction(1, 4) and conditional == 0

# Caution 3: one-bit fingerprints, k = 2, m = n = 2. Enumerate placements, the slot winner
# (uniform among the sharers), and both fingerprints; member 0 is the query.
fnr_fp = Fraction(0)
p_disp = Fraction(0)
for pl in placements:
    sharers = [i for i in range(2) if pl[i] == pl[0]]
    for winner in sharers:
        w = Fraction(1, 4) * Fraction(1, len(sharers))
        for g0, g1 in product(range(2), repeat=2):
            stored = (g0, g1)[winner]
            weight = w * Fraction(1, 4)
            p_disp += weight * (winner != 0)
            fnr_fp += weight * (stored != g0)
print(f'one-bit fingerprints, k = 2: displacement probability {p_disp}, membership FNR {fnr_fp} '
      f'= {p_disp} * (1 - 2^-1)')
assert p_disp == Fraction(1, 4) and fnr_fp == Fraction(1, 8)

182 two-member slots: in every one exactly one member is displaced (miss indicators sum to 1, correlation -1)
balanced packing at m = 900: misses = 100 in every build (variance 0); independent coins at 0.10 would have variance 90
m = n = 2, k = 1: unconditional expected displaced fraction = 1/4; conditional on the build rejecting overfull slots = 0
one-bit fingerprints, k = 2: displacement probability 1/4, membership FNR 1/8 = 1/4 * (1 - 2^-1)


## Claim 5: ranked search (top-$k$ over a corpus) is the witness model

Build the BPHF over the whole corpus with each document's sort position as its rank ($0$ for the top score). Top-$k$ retrieval is a rank-bound check, $\textsc{TopK}(a) = \textsc{Query}(a) \land R[\phi(a)] < k$. A corpus document carries its rank; an off-corpus candidate has none to present, so nothing outside the corpus is ever surfaced. At full resolution the retrieval is exact. Under a $k$-perfect hash with $m < n$ fully occupied slots, which documents are displaced is independent of their rank (the perfect hash sees identifiers, not scores), so by linearity the expected number of missed top-$k$ documents is $k(1 - m/n)$ (`prop:bphf-ranked-search`), whatever the dependence among the misses.

We run one structural $k$-perfect build at $m = 0.99\,n$, then average the miss count over many builds, each displacing a uniformly random set of $n - m$ documents.

In [8]:
N = 100_000
k = 100
scores = rng.uniform(size=N)
order = np.argsort(-scores, kind='stable')
docs = [f'doc{i:06d}' for i in range(N)]
corpus_rank = {docs[int(i)]: pos for pos, i in enumerate(order)}   # 0 = top score
top = [docs[int(i)] for i in order[:k]]                           # the true top-k documents

search = BPHF(docs, corpus_rank)


def top_k(engine, d, presented):
    return engine.query(d, presented) and engine.R[engine._phi_ext(d)] < k


off_corpus = [f'other{j}' for j in range(20_000)]                 # not in the corpus, no valid rank
recall = sum(top_k(search, d, lambda z: corpus_rank[z]) for d in top)
surfaced = sum(top_k(search, d, lambda z: -1) for d in off_corpus)
spurious = sum(top_k(search, docs[int(i)], lambda z: corpus_rank[z]) for i in order[k:k + 5000])
print(f'exact corner: top-{k} recall {recall}/{k}; off-corpus surfaced {surfaced}; '
      f'corpus documents below rank {k} reported in the top {k}: {spurious}')
assert recall == k and surfaced == 0 and spurious == 0


class KPerfectBPHF(BPHF):
    # A k-perfect build: m slots, all occupied; n - m members share a slot and lose it.
    def __init__(self, members, ranks, m, rng_build):
        perm = rng_build.permutation(len(members))
        self.n = m
        self.phi = {}
        self.R = [None] * m
        for pos, idx in enumerate(perm):
            mem = members[int(idx)]
            slot = pos if pos < m else int(rng_build.integers(0, m))   # owners first, then sharers
            self.phi[mem] = slot
            if pos < m:
                self.R[slot] = ranks[mem]


rng_build = np.random.default_rng(1516)
frac = 0.99
kp = KPerfectBPHF(docs, corpus_rank, int(frac * N), rng_build)
misses_one = k - sum(top_k(kp, d, lambda z: corpus_rank[z]) for d in top)
surfaced_kp = sum(top_k(kp, d, lambda z: -1) for d in off_corpus)
print(f'one k-perfect build at m = {frac:.2f} n: {misses_one} of the top {k} missed; off-corpus surfaced {surfaced_kp}')
assert surfaced_kp == 0, 'FPR stays 0 under the k-perfect dial'

top_idx = order[:k]
for frac in (1.00, 0.99, 0.95):
    m = int(round(frac * N))
    builds = 1000
    misses = np.array([np.isin(top_idx, rng_build.choice(N, size=N - m, replace=False)).sum()
                       for _ in range(builds)])
    expected = k * (1 - m / N)
    se = misses.std(ddof=1) / math.sqrt(builds) if misses.std() > 0 else 0.0
    print(f'  m = {frac:.2f} n: mean top-{k} misses over {builds} builds = {misses.mean():.2f} '
          f'(SE {se:.2f});  k(1 - m/n) = {expected:.1f};  FPR = 0')
    assert abs(misses.mean() - expected) <= max(4 * se, 1e-12)

exact corner: top-100 recall 100/100; off-corpus surfaced 0; corpus documents below rank 100 reported in the top 100: 0


one k-perfect build at m = 0.99 n: 0 of the top 100 missed; off-corpus surfaced 0
  m = 1.00 n: mean top-100 misses over 1000 builds = 0.00 (SE 0.00);  k(1 - m/n) = 0.0;  FPR = 0


  m = 0.99 n: mean top-100 misses over 1000 builds = 1.04 (SE 0.03);  k(1 - m/n) = 1.0;  FPR = 0


  m = 0.95 n: mean top-100 misses over 1000 builds = 5.01 (SE 0.07);  k(1 - m/n) = 5.0;  FPR = 0


## Claim 6: space, and the crossover at $\fprate = 1/n$

`prop:bphf-space`: the BPHF uses $\lceil \log_2 n \rceil + c_\phi$ bits per element, where $c_\phi \ge \log_2 e - o(1)$ is the minimal perfect hash's addressing cost; the perfect hash filter uses $\lceil -\log_2 \fprate \rceil + c_\phi$. Both inherit the same perfect hash, so they share the addressing term and differ only in the slot width: the PHF's is set by the target rate, the BPHF's by the set size. The widths tie at $\fprate = 1/n$. Above it (looser targets) the PHF is cheaper; below it the BPHF is, and its extra bits in the loose regime buy a guarantee (no false positives from candidates without a valid rank) that no fingerprint width delivers.

We read $c_\phi$ off the library's BBHash-backed perfect hash filter on the same $n = 1000$ members, reproduce the chapter's worked example (about $11{,}500$ to $13{,}300$ bits in total for the BPHF at $n = 1000$, for $c_\phi$ between $1.5$ and $3.3$), and the slot-width table at $n = 10^6$ (`tab:bhf-bphf-bits`).

In [9]:
LOG2E = math.log2(math.e)
c_phi = pf.bits_per_key() - pf.fingerprint_bits()       # BBHash addressing cost on these members
print(f'addressing cost c_phi = {c_phi:.2f} bits/key (>= log2 e = {LOG2E:.2f}), shared by both structures')
assert c_phi >= LOG2E

bphf_bpe = r + c_phi
print(f'BPHF at n = {n}: {r} + {c_phi:.2f} = {bphf_bpe:.2f} bits/element, {n * r} bits of ranks + '
      f'{n * c_phi:.0f} bits of perfect hash = {n * bphf_bpe:.0f} bits')
lo, hi = n * (r + 1.5), n * (r + 3.3)
print(f'  the chapter\'s range for c_phi in [1.5, 3.3]: {lo:.0f} to {hi:.0f} bits')
assert lo <= n * bphf_bpe <= hi

print()
for eps in (0.1, 0.01, 1.0 / n, 1e-4):
    r_phf = int(math.ceil(-math.log2(eps)))
    phf_bpe = r_phf + c_phi
    rel = 'BPHF < PHF' if bphf_bpe < phf_bpe else ('equal' if bphf_bpe == phf_bpe else 'PHF < BPHF')
    print(f'FPR = {eps:8.5f}: BPHF {bphf_bpe:5.2f} bits/element, PHF {phf_bpe:5.2f}  ({rel})')

r_cross = int(math.ceil(-math.log2(1.0 / n)))
print(f'crossover: r_PHF(1/n) = {r_cross} = r_BPHF = {r}')
assert r_cross == r, 'PHF and BPHF slot widths must tie at FPR = 1/n'

print('\nslot widths at n = 10^6 (both omit the shared c_phi):')
n6 = 10 ** 6
r_bphf6 = math.ceil(math.log2(n6))
table = {eps: math.ceil(-math.log2(eps)) for eps in (1e-2, 1e-3, 1e-6)}
for eps, w in table.items():
    print(f'  FPR = {eps:.0e}: PHF {w:>2}   BPHF {r_bphf6}')
assert list(table.values()) == [7, 10, 20] and r_bphf6 == 20
print(f'  extra BPHF bits at FPR = 1e-2: {r_bphf6 - table[1e-2]} per element, the price of turning '
      f'a 1e-2 probability into a certainty in the witness model')
assert r_bphf6 - table[1e-2] == 13
print('\nAll chapter 15 claims verified.')

addressing cost c_phi = 3.26 bits/key (>= log2 e = 1.44), shared by both structures
BPHF at n = 1000: 10 + 3.26 = 13.26 bits/element, 10000 bits of ranks + 3258 bits of perfect hash = 13258 bits
  the chapter's range for c_phi in [1.5, 3.3]: 11500 to 13300 bits

FPR =  0.10000: BPHF 13.26 bits/element, PHF  7.26  (PHF < BPHF)
FPR =  0.01000: BPHF 13.26 bits/element, PHF 10.26  (PHF < BPHF)
FPR =  0.00100: BPHF 13.26 bits/element, PHF 13.26  (equal)
FPR =  0.00010: BPHF 13.26 bits/element, PHF 17.26  (BPHF < PHF)
crossover: r_PHF(1/n) = 10 = r_BPHF = 10

slot widths at n = 10^6 (both omit the shared c_phi):
  FPR = 1e-02: PHF  7   BPHF 20
  FPR = 1e-03: PHF 10   BPHF 20
  FPR = 1e-06: PHF 20   BPHF 20
  extra BPHF bits at FPR = 1e-2: 13 per element, the price of turning a 1e-2 probability into a certainty in the witness model

All chapter 15 claims verified.


## Closing: what this notebook verified

| Claim | Statement | Verified |
|-------|-----------|----------|
| 1 | The bare BPHF over the whole universe is the perfect hash filter at $r = \lceil \log_2 n \rceil$: $\fnrate = 0$, $\fprate \approx 2^{-r}$ | yes (asserted, against `PerfectFilter` at the same $r$) |
| 2 | Witness model: $\fprate = 0$ exactly; a constant or public low-entropy rank is forgeable | yes (asserted) |
| 3 | True perfect hash: no member misses, even with rank collisions | yes (asserted) |
| 4 | $k$-perfect dial: $\fnrate = 1 - o/n \ge 1 - m/n$; misses dependent within slots; conditioning matters ($1/4$ vs $0$); fingerprint rescue ($1/8$) | yes (asserted, exact enumeration) |
| 5 | Ranked search: exact at full resolution, $k(1 - m/n)$ expected misses, no off-corpus result | yes (asserted) |
| 6 | Space $\lceil \log_2 n \rceil + c_\phi$ with $c_\phi \ge \log_2 e$; crossover with the PHF at $\fprate = 1/n$ | yes (asserted) |

The two arms of the perfect-filter line are not symmetric in cost. A false positive is a coincidence, one stored value colliding with one recomputed value, and a single $r$-bit slot makes it rare. A guarantee of no false positives is a certificate enforced against every non-member at once; the BPHF obtains it only by restricting the query model, and a universe-wide FPR-zero filter would cost about $\log_2(|\Universe|/n)$ bits per element, the price of a dictionary. Chapter 16 takes up the adversarial query that the witness model must exclude.